# Tanimoto similarity between molecules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/comp_chem_colab/blob/main/Tanimoto_similarity.ipynb)

The Tanimoto coefficient compares two molecular fingerprints: the number of bits set in
both, divided by the number of bits set in either. It runs from 0 (nothing in common) to
1 (identical fingerprints). This notebook computes it for pairs of molecules given as
SMILES, then for a whole set at once, using Morgan (circular) fingerprints from RDKit.

Runtime: CPU. Takes seconds.

## Setup

RDKit is not preinstalled on Colab; pandas, NumPy, scikit-learn and Matplotlib are.

In [ ]:
%pip install -q rdkit

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, rdFingerprintGenerator

## Similarity of two molecules

A Morgan fingerprint of radius 3 (similar to ECFP6) folded to 2048 bits. A smaller radius
(2, similar to ECFP4) captures smaller fragments and usually gives higher similarities for
the same pair, so only compare values computed with the same settings.

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)


def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit cannot read this SMILES: {smiles}")
    return morgan.GetFingerprint(mol)


def tanimoto(smiles1, smiles2):
    return round(DataStructs.TanimotoSimilarity(fingerprint(smiles1), fingerprint(smiles2)), 3)

In [ ]:
molecules = {
    "mol1": "C1OC2=C(O1)C=C(C=C2)C3=CC(=NN3)C4=CC(=CC=C4)Br",
    "mol2": "CC1=CC=C(C=C1)CC2(C3=C(C(=CC=C3)Cl)NC2=O)C",
    "mol3": "CCCCC(=O)NC(=S)NNC(=O)C1=CC=C(C=C1)OC",
    "mol4": "C1=CC=C2C=C(C=CC2=C1)S(=O)(=O)NC3=CC=CC4=C3N=CC=C4",
    "mol5": "C1=CC=C(C=C1)CCC(=O)NNC(=S)NC(=O)C2=CC=C(C=C2)Cl",
    "mol6": "COc1cc2occc2c2oc(-c3ccc4c(c3)OCO4)cc(=O)c12",
    "mol7": "c1ccc(-c2ncc(-c3ccc4c(c3)OCO4)o2)cc1",
    "mol8": "OC(/C=C/c1cccc2cccnc12)=NN=C(O)c1cc(-c2ccccc2)on1",
    "mol9": "CC(c1c[nH]c2ccccc12)c1c[nH]c2ccccc12",
    "mol10": "c1ccc2ncc(-c3c[nH]c4ccccc34)cc2c1",
}

tanimoto(molecules["mol1"], molecules["mol6"])

In [ ]:
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in molecules.values()],
                     legends=list(molecules), molsPerRow=5, subImgSize=(220, 180))

## Similarity matrix of the whole set

`BulkTanimotoSimilarity` compares one fingerprint with a list of them in a single call,
which is much faster than a Python loop over pairs for large sets.

In [ ]:
names = list(molecules)
fps = [fingerprint(s) for s in molecules.values()]
matrix = np.array([DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=90)
ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="black" if matrix[i, j] > 0.5 else "white")
fig.colorbar(im, label="Tanimoto similarity")
plt.tight_layout()
plt.show()

In [ ]:
# The most similar distinct pairs
pairs = [(matrix[i, j], names[i], names[j])
         for i in range(len(names)) for j in range(i + 1, len(names))]
for score, a, b in sorted(pairs, reverse=True)[:5]:
    print(f"{a:>5} {b:>5}  {score:.3f}")